In [0]:
import json, os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, refresh
from pyspark.sql import functions as F

# t31_refresh: after a controlled refresh (5.11), in the target (test or dev). Read-only.
# Refreshed rows carry re-keyed ids (<letters>R<10 hex>). Checks: they are all there, people are faked (Aadhaar and phone
# start with 0, name "Person ..."), and every refreshed child row has its refreshed parent.
dbutils.widgets.dropdown("env", "test", ["test", "dev"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
cat = cfg["catalog"]
rcfg, parents, ids = refresh.load()
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print(("PASS " if ok else "FAIL ") + name + (f" -> {detail}" if detail and not ok else ""))

base = f"/Volumes/{cat}/ops/refresh_in"
run = dbutils.fs.head(f"{base}/_latest").strip()
man = json.loads(dbutils.fs.head(f"{base}/{run}/_manifest.json"))
print("INFO run", run, "from", man["from"], f"({man['pct']}% of people)")
RE = r"^[A-Za-z_]*R[0-9A-F]{10}$"
s = lambda t: spark.table(f"{cat}.silver.{t}")
for t in refresh.order(parents):
    key = parents[t][1][0]
    if key in ids and t in man["rows"]:
        n = s(t).filter(F.col(key).rlike(RE)).count()
        check(f"{t}: refreshed rows present", n >= man["rows"][t] > 0 or man["rows"][t] == 0, f"{n} of {man['rows'][t]}")
p = rcfg["person"]
people = s(p["table"]).filter(F.col(p["key"]).rlike(RE))
fakes = rcfg["fake"][p["table"]]
real = people.filter(" OR ".join(
    [f"({c} IS NOT NULL AND NOT {c} LIKE '0%')" for c, k in fakes.items() if k in ("aadhaar", "phone")] +
    [f"({c} IS NOT NULL AND NOT {c} LIKE 'Person %')" for c, k in fakes.items() if k == "name"])).count()
check("refreshed people are faked (Aadhaar/phone start with 0, name Person ...)", real == 0, real)
check("refreshed people have a fingerprint made with this environment's key",
      people.filter(f"{p['sample_by']} IS NOT NULL AND id_fingerprint IS NULL").count() == 0)
for t, (par, _) in parents.items():
    for c, pt in par.items():
        pk = parents[pt][1][0]
        if c in ids:
            lost = (s(t).filter(F.col(c).rlike(RE)).select(c)
                     .join(s(pt).select(F.col(pk).alias(c)), c, "left_anti").count())
            check(f"{t}.{c}: every refreshed row has its parent", lost == 0, lost)
print(f"\nt31_refresh: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t31_refresh failed")